# Comparing Downsampling Strategies for the Classic CNN Reconstructor

Every reconstructor architecture in this repo that pools at all -- `SimpleNet`, `Rama`, `Papyrus1stStage`/`Papyrus2ndStage` (`AI4AO/PhaseEstimators.py`), `ClassicCNN` and its siblings in `CNNArchitectureComparison.ipynb`, and the variants in `NormalizationComparison.ipynb`/`ActivationComparison.ipynb` -- downsamples with plain `nn.MaxPool2d(2)`: a hard max over each 2x2 window that discards 3 of every 4 samples with no way for the network to recover them later. This notebook holds `ClassicCNN` fixed and sweeps only *which operation halves spatial size between conv stages*, following the exact same procedure `NormalizationComparison.ipynb`/`ActivationComparison.ipynb` already establish (`AI4AO.Trainer.Trainer`, same instrument, same loss, same `TrainRunNb`), comparing training-loss curves and seeded closed-loop steady-state residual wavefront error (nm RMS). See `Ideas/13-pooling-downsampling-comparison.md` for the full design discussion and the Q&A that resolved its open forks.

**The three variants:**

1. **`MaxPool`** -- the existing baseline, unmodified: channels grow via conv, `nn.MaxPool2d(2)` halves spatial size afterward.
2. **`SpaceToDepth`** -- `nn.PixelUnshuffle(2)`: each 2x2 spatial block's 4 pixels become 4 separate channels, per input channel. A raw pixel reshuffle -- bijective (invertible), so nothing is discarded. This is the same operation behind YOLOv5's "Focus" layer and the "squeeze" step in RealNVP/Glow-style normalizing flows.
3. **`Haar`** -- a single-level 2D Haar wavelet decomposition (average, vertical-detail, horizontal-detail, diagonal-detail), also bijective, but an orthonormal change of basis rather than an arbitrary pixel permutation -- explicit per-orientation frequency content instead of 4 relabeled pixels.

**Why this might matter, and why it's not free.** Max/average pooling is a many-to-one, non-invertible map -- a textbook source of aliasing (the reason anti-aliased-downsampling work like BlurPool exists, low-pass filtering *before* subsampling to fight it). `SpaceToDepth`/`Haar` don't combine any pixels at all, so they structurally cannot alias. But "nothing is lost" is only true at that one layer: per the Q&A in `Ideas/13`, this notebook deliberately runs both bijective variants **without a channel-projection conv** after each downsample, so channels compound ~4x/stage instead of being reduced back down -- on purpose, to test the "nothing thrown away" hypothesis in its purest form. The cost: `SpaceToDepth`/`Haar` are capped at 2 downsampling stages instead of `MaxPool`'s 4 (deeper would be impractical on a single GPU -- see the arithmetic below), so the three variants end up with different parameter counts *and* different final spatial resolution before the shared head. Both differences are surfaced explicitly below and in `Ideas/13`, not tuned away.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

from AI4AO.paths import DATA_DIR
from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'


## Configuration and instrument

Reuses `CNNComparison_params.py` unchanged -- the same single fictional-demo, nominal-geometry modulated Pyramid WFS + DM (no real bench), so this comparison runs under identical atmosphere/noise/DM/loss conditions as the architecture and normalization comparisons, differing only in which axis (downsampling operation) is swept. Checkpoints are kept separate, under their own `Data/PoolingComparison/` directory.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "PoolingComparison"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")


## The downsampling building blocks

`maxpool_stage` is `ClassicCNN`'s original stage, unmodified. `HaarDownsample2d` is the one genuinely new component here -- a fixed (non-learned), orthonormal single-level 2D Haar wavelet downsample implemented as a grouped `F.conv2d` with a fixed weight buffer, so it's usable as a drop-in alternative to `nn.PixelUnshuffle(2)`. `MaintainThenDownsample` is the block implementing the original idea directly: two same-channel 3x3 convs (no channel growth in the conv itself), then a bijective downsample that turns each channel into 4. `pad_to_even` exists because `nn.PixelUnshuffle` and a stride-2 conv misbehave *differently* on odd spatial input -- `PixelUnshuffle` raises a hard error, a stride-2 conv silently truncates -- so padding first avoids both, identically for every variant.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can take before
    dropping below min_size pixels -- same helper as CNNArchitectureComparison.ipynb's
    ClassicCNN, reused unmodified here for the MaxPool baseline's depth."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def pad_to_even(x):
    """Zero-pads the spatial dims of an (B,C,H,W) tensor by 1 pixel wherever H or W is
    odd, so a stride-2 bijective downsample always sees an even input. Needed because the
    two variants below misbehave differently on odd input otherwise: PixelUnshuffle raises
    a hard RuntimeError, while a stride-2 conv (HaarDownsample2d) silently truncates the
    trailing row/column -- padding first avoids both."""
    _, _, H, W = x.shape
    pad_h, pad_w = H % 2, W % 2
    if pad_h or pad_w:
        x = F.pad(x, (0, pad_w, 0, pad_h))
    return x


class HaarDownsample2d(nn.Module):
    """Fixed (non-learned), orthonormal single-level 2D Haar wavelet downsample: every
    input channel's 2x2 blocks are decomposed into 4 output channels -- LL (average), LH
    (row/vertical detail), HL (column/horizontal detail), HH (diagonal detail) -- via a
    grouped conv with a fixed weight. Unlike nn.PixelUnshuffle's raw pixel reshuffle, the 4
    output channels are an orthonormal change of basis (each 2x2 filter has unit norm and
    is orthogonal to the other three), so this separates frequency/orientation content
    explicitly instead of just relabeling the same 4 pixels as channels. Still bijective
    (invertible, energy-preserving by Parseval's theorem), and the gradient w.r.t. the
    input stays intact since the fixed kernel is applied via F.conv2d."""

    def __init__(self):
        super().__init__()
        kernels = torch.tensor([
            [[0.5, 0.5], [0.5, 0.5]],
            [[0.5, 0.5], [-0.5, -0.5]],
            [[0.5, -0.5], [0.5, -0.5]],
            [[0.5, -0.5], [-0.5, 0.5]],
        ])
        self.register_buffer("kernels", kernels)

    def forward(self, x):
        C = x.shape[1]
        weight = self.kernels.unsqueeze(1).repeat(C, 1, 1, 1).to(x.dtype)  # (4C, 1, 2, 2)
        return F.conv2d(x, weight, stride=2, groups=C)


def maxpool_stage(in_channels, out_channels, activation=nn.LeakyReLU):
    """ClassicCNN's original stage, unmodified: channels grow via the convs themselves,
    then MaxPool2d(2) halves spatial size, discarding 3 of every 4 samples per window via a
    hard max -- the baseline this whole notebook argues about."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1), activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1), activation(),
        nn.MaxPool2d(2),
    )

def avgpool_stage(in_channels, out_channels, activation=nn.LeakyReLU):
    """ClassicCNN's original stage, unmodified: channels grow via the convs themselves,
    then MaxPool2d(2) halves spatial size, discarding 3 of every 4 samples per window via a
    hard max -- the baseline this whole notebook argues about."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1), activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1), activation(),
        nn.AvgPool2d(2),
    )


class MaintainThenDownsample(nn.Module):
    """The proposed alternative to maxpool_stage: two same-channel 3x3 convs (channel
    count held fixed -- all the feature growth happens in the downsample, not the convs)
    followed by a bijective spatial downsample that turns each channel into 4, quartering
    spatial resolution without discarding any value the convs computed. Per Ideas/13's Q&A,
    no projection conv follows the downsample -- channel count is left to compound ~4x per
    stage, deliberately, to test the 'nothing is thrown away' hypothesis in its purest
    form."""

    def __init__(self, channels, downsample_module, activation=nn.LeakyReLU):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.act = activation()
        self.downsample = downsample_module

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = pad_to_even(x)
        return self.downsample(x)


DOWNSAMPLE_FACTORIES = {
    "SpaceToDepth": lambda: nn.PixelUnshuffle(2),
    "Haar": lambda: HaarDownsample2d(),
}

# 4x/stage channel compounding (no projection, per the Q&A) makes MaxPool's own depth
# (num_downsamples(Nout), 4 stages at the current Nout) impractical for the two bijective
# variants -- see the notebook intro and Ideas/13 for the arithmetic. Capped separately here.
POOL_STAGES_BIJECTIVE = 3


Two cheap numerical checks before trusting either bijective downsample in training: `HaarDownsample2d`'s kernel is truly orthonormal (so it's actually the invertible, energy-preserving transform it's supposed to be, not just "shaped like" a wavelet decomposition), and gradients reach the input through both `SpaceToDepth` and `Haar` -- the same "confirmed in code, still worth a standing numeric check" convention `CNNArchitectureComparison.ipynb`'s `PatchEmbed` pupil-isolation check follows.

In [ ]:
# Orthonormality check: HaarDownsample2d's 4 (2x2) filters, flattened, should form an
# orthonormal basis of R^4 (each unit norm, all pairwise dot products zero) -- this is what
# makes the transform truly invertible and energy-preserving, not just "shaped like" a
# wavelet decomposition.
_haar_check = HaarDownsample2d()
_flat = _haar_check.kernels.reshape(4, 4)
_gram = _flat @ _flat.T
assert torch.allclose(_gram, torch.eye(4), atol=1e-6), f"Haar kernels not orthonormal:\n{_gram}"
print("HaarDownsample2d kernel orthonormality check passed.")

# Differentiability check: gradient must reach the input through both bijective downsamples.
_x = torch.randn(2, 4, 8, 8, requires_grad=True)
for _name, _factory in DOWNSAMPLE_FACTORIES.items():
    _out = _factory()(_x)
    _out.sum().backward()
    assert _x.grad is not None and torch.isfinite(_x.grad).all(), f"{_name} broke gradient flow"
    _x.grad = None
print("SpaceToDepth / Haar gradient-flow check passed.")


## The `PoolingCNN` architecture

`num_downsamples`/`base_channels=28` mirror `ClassicCNN` exactly for the `MaxPool` baseline. For `SpaceToDepth`/`Haar`, an initial stem conv lifts the 4 pupil channels to `base_channels`, then each of `POOL_STAGES_BIJECTIVE` stages applies `MaintainThenDownsample` -- channels held fixed through the convs, quadrupled only by the downsample. Holding every other choice fixed (conv kernel sizes, activation, global-average-pool + linear head) isolates the comparison to the downsampling operation alone.

In [ ]:
class PoolingCNN(nn.Module):
    """ClassicCNN (CNNArchitectureComparison.ipynb) with one added axis of variation:
    which downsampling operation follows each pair of convs. 'MaxPool' reproduces
    ClassicCNN's stage exactly (channels grow via conv, MaxPool2d(2) after).
    'SpaceToDepth'/'Haar' instead hold each stage's conv channels fixed and grow channels
    only through a bijective 2x2 -> 4-channel spatial downsample (raw pixel reshuffle vs. a
    fixed orthonormal Haar decomposition). Per Ideas/13's design discussion, MaxPool and
    the other two variants do NOT share a channel/depth schedule or a matched parameter
    budget -- that's a disclosed, deliberate consequence of the 'no projection after the
    downsample' choice, not an oversight. Treat the printed parameter counts and final
    spatial resolution (see the instantiation cell below) as things to report, not
    pre-equalized."""

    def __init__(self, n_channels, Nmodes, Nout, downsample_name, base_channels=28,
                 pool_stages=POOL_STAGES_BIJECTIVE):
        super().__init__()
        self.downsample_name = downsample_name

        if downsample_name == "MaxPool":
            n_stages = num_downsamples(Nout)
            channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]
            layers = [maxpool_stage(channels[i], channels[i + 1]) for i in range(n_stages)]
            final_channels = channels[-1]
        elif downsample_name == "AvgPool":
            n_stages = num_downsamples(Nout)
            channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]
            layers = [avgpool_stage(channels[i], channels[i + 1]) for i in range(n_stages)]
            final_channels = channels[-1]
        else:
            downsample_factory = DOWNSAMPLE_FACTORIES[downsample_name]
            layers = [nn.Conv2d(n_channels, base_channels, kernel_size=3, padding=1), nn.LeakyReLU()]
            channels = base_channels
            for _ in range(pool_stages):
                layers.append(MaintainThenDownsample(channels, downsample_factory()))
                channels *= 4
            final_channels = channels

        layers.append(nn.AdaptiveAvgPool2d(1))
        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(final_channels, Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


## Instantiating all three variants

Each variant is built once here and printed with its total trainable parameter count. Unlike `CNNArchitectureComparison.ipynb`'s matched-budget six architectures, `MaxPool` and the two bijective variants are **not** sized to the same parameter count -- `SpaceToDepth`/`Haar` end up with roughly a quarter of `MaxPool`'s trainable parameters despite finishing with *more* feature channels (448 vs. 224 at the current `Nout`/`base_channels`), precisely because their channel growth comes for free from the downsample op rather than from learned conv weights. That asymmetry -- and the final-spatial-resolution difference it comes with (~12x12 vs. ~2x2 into the shared `AdaptiveAvgPool2d(1)`) -- is itself part of what this comparison is testing; see `Ideas/13` for the full arithmetic.

In [ ]:
architectures = {
    name: PoolingCNN(n_channels, Nmodes, Nout, downsample_name=name, base_channels=32).to(device=device)
    for name in ["MaxPool", "AvgPool"]
}

for name, model in architectures.items():
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:12s} -- total trainable parameters: {total_params:,}")


## Training

Following `NormalizationComparison.ipynb`'s exact procedure, repeated once per downsampling variant: a fresh `AdamW` optimizer and a fresh `Trainer` per variant, all sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)`.

Same per-variant-independent-draws trade-off as every other notebook in this family: `Trainer.train()` redraws `self.dataset[0]` fresh at every outer step regardless of which `Trainer` instance calls it, so the three variants train on independent i.i.d. draws from the same distribution rather than paired ones -- the training curves below are directionally informative only; the seeded closed-loop rollout further down is the true apples-to-apples comparison.

Three variants is cheaper than `CNNArchitectureComparison.ipynb`'s six, but still three full training runs on a single memory-constrained GPU -- consider a smaller `TrainParams['TrainRunNb']` for an initial correctness pass before committing to the full training budget.

In [ ]:
# Closed-loop (BPTT) iterations per optimizer step in the second training stage. Stage 1 uses
# TrainParams['ClosedLoopIterations'] from CNNComparison_params.py (a single pass); this is the
# one training setting that is not in that shared file, so it lives here.
CLOSED_LOOP_ITERATIONS_STAGE2 = 10

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 1: single pass) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this variant -- e.g. after editing base_channels or
        # POOL_STAGES_BIJECTIVE above, old checkpoints under PATH are no longer compatible
        # and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


### Stage 2 -- closed-loop training

The loop above (stage 1) trains every model with `TrainParams['ClosedLoopIterations']` from `CNNComparison_params.py`, i.e. a single pass per optimizer step. The next cell repeats the identical loop with `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per optimizer step, continuing from the weights stage 1 just saved (`load_checkpoint` at the top of the loop, and a fresh `AdamW` per model, as in stage 1).

The checkpoint under `PATH` is overwritten with the stage-2 weights, and from here on `trainers`/`loss_trackers`/`loss_trackers_ideal` -- the ones the plotting and evaluation cells below use -- refer to stage 2.

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 2: closed loop) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this variant -- e.g. after editing base_channels or
        # POOL_STAGES_BIJECTIVE above, old checkpoints under PATH are no longer compatible
        # and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], CLOSED_LOOP_ITERATIONS_STAGE2)

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


## Comparing training-loss curves

`Trainer.plot_losses` only plots one tracker pair at a time, so we reimplement just its smoothing-and-overlay logic here for all three trackers at once. The oracle "ideal loss" bound (`z_inv` applied to the true residual OPD) is a property of the atmosphere/noise distribution and the DM, not of the downsampling choice, so the three `loss_tracker_ideal` curves are statistically equivalent -- only one is shown, as a reference lower bound shared by all three.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for name, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=name)

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="--")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Loss")
ax.legend()
plt.show()


## Comparing closed-loop residual wavefront error (nm RMS)

For each variant, reseed immediately before the rollout so all three see identical wind/r0/noise draws, then run `trainer.evaluate()` and compute the steady-state residual wavefront error in nm RMS over the pupil, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()` uses internally).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator warm-up
    period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")


In [ ]:
names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]
param_counts = [sum(p.numel() for p in architectures[n].parameters() if p.requires_grad) for n in names]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].bar(names, rms_values)
axes[0].set_ylabel("Steady-state residual (nm RMS)")
axes[0].set_title("Closed-loop performance")
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, param_counts)
axes[1].set_ylabel("Trainable parameters")
axes[1].set_title("Model size")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## Visualizing the best downsampling variant's closed loop

A closer look at whichever variant came out on top above -- the same rollout-animation pattern `CNNArchitectureComparison.ipynb`/`NormalizationComparison.ipynb` use.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best downsampling variant: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())
